# StemScribe Guitar Transcription Training
## Domain Adaptation: Kong Piano -> GuitarSet

**Goal:** Fine-tune Kong piano checkpoint on GuitarSet to achieve 87%+ onset F1 (vs 79% Basic Pitch)

**Runtime:** Select A100 GPU (Runtime > Change runtime type > A100)

**Time:** ~2h Phase 1 + ~6h Phase 2 = ~8h total on A100

**Instructions:**
1. Connect to A100 GPU runtime
2. Run all cells in order
3. Model auto-saves to Google Drive every 5 epochs
4. Final model: `drive/MyDrive/stemscribe_checkpoints/best_guitar_model.pt`

In [ ]:
# Cell 1: Mount Google Drive for checkpoint persistence
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_CHECKPOINT_DIR = '/content/drive/MyDrive/stemscribe_checkpoints'
os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)
print(f'Checkpoints will save to: {DRIVE_CHECKPOINT_DIR}')

In [ ]:
# Cell 2: Check GPU
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
else:
    print('WARNING: No GPU! Go to Runtime > Change runtime type > A100')

In [ ]:
# Cell 3: Install dependencies
!pip install -q librosa jams mir_eval pretty_midi tqdm audiomentations soundfile

In [ ]:
# Cell 4: Download Kong piano checkpoint from Zenodo (172 MB)
!wget -q --show-progress -O /content/kong_piano_checkpoint.pth \
    'https://zenodo.org/record/4034264/files/CRNN_note_F1%3D0.9677_pedal_F1%3D0.9186.pth'
print(f'Kong checkpoint: {os.path.getsize("/content/kong_piano_checkpoint.pth") / 1e6:.1f} MB')

In [ ]:
# Cell 5: Download GuitarSet from Zenodo
!mkdir -p /content/guitarset
!wget -q --show-progress -O /content/guitarset/audio_mono-mic.zip \
    'https://zenodo.org/record/3371780/files/audio_mono-mic.zip'
!wget -q --show-progress -O /content/guitarset/annotation.zip \
    'https://zenodo.org/record/3371780/files/annotation.zip'
!cd /content/guitarset && unzip -qo audio_mono-mic.zip && unzip -qo annotation.zip
!rm -f /content/guitarset/*.zip
!echo "Audio files:" && ls /content/guitarset/audio_mono-mic/*.wav | wc -l
!echo "Annotation files:" && ls /content/guitarset/annotation/*.jams | wc -l

In [ ]:
# Cell 6: Upload training script from local repo
# Option A: Upload train_guitar_runpod.py via Colab file upload
# Option B: Paste inline (done below)

# We'll download the training script content inline to avoid upload issues.
# This is the same as train_guitar_model/train_guitar_runpod.py but with
# paths adjusted for Colab.

print('Training script will be defined in the next cell.')
print('Using /content/ paths instead of /workspace/')

In [ ]:
# Cell 7: Override paths for Colab environment
import shutil

# Colab paths (overriding RunPod paths)
DATA_DIR_COLAB = '/content/guitarset'
SAVE_DIR_COLAB = '/content/guitar_results'
KONG_PATH_COLAB = '/content/kong_piano_checkpoint.pth'

os.makedirs(SAVE_DIR_COLAB, exist_ok=True)
os.makedirs(f'{SAVE_DIR_COLAB}/checkpoints', exist_ok=True)

# Google Drive checkpoint sync function
def sync_to_drive(local_path, drive_dir=DRIVE_CHECKPOINT_DIR):
    """Copy checkpoint to Google Drive for persistence."""
    fname = os.path.basename(local_path)
    dst = os.path.join(drive_dir, fname)
    shutil.copy2(local_path, dst)
    print(f'  Synced to Drive: {dst}')

print('Colab environment configured.')

In [ ]:
%%writefile /content/train_guitar_colab.py
#!/usr/bin/env python3
"""
Guitar Transcription Training - Colab Version
Domain Adaptation from Kong Piano Checkpoint -> GuitarSet
"""

import os
import sys
import time
import random
import shutil
import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
import librosa
import jams
import mir_eval
from tqdm import tqdm

# ============================================================================
# CONFIG
# ============================================================================
CONFIG = {
    'sample_rate': 16000,
    'hop_length': 160,
    'n_fft': 2048,
    'n_mels': 229,
    'guitar_midi_min': 40,
    'guitar_midi_max': 88,
    'n_pitches': 48,
    'chunk_seconds': 10.0,
    'phase1_epochs': 20,
    'phase1_lr': 5e-5,
    'phase2_epochs': 80,
    'phase2_lr': 1e-5,
    'batch_size': 4,
    'onset_pos_weight': 50.0,
    'frame_pos_weight': 10.0,
    'focal_gamma': 2.0,
    'augment': True,
    'gain_range': (-6, 6),
    'noise_snr_range': (20, 40),
    'spec_augment_freq_masks': 2,
    'spec_augment_time_masks': 2,
    'spec_augment_freq_width': 15,
    'spec_augment_time_width': 40,
}

SAMPLE_RATE = CONFIG['sample_rate']
HOP_LENGTH = CONFIG['hop_length']
N_MELS = CONFIG['n_mels']
N_PITCHES = CONFIG['n_pitches']
GUITAR_MIDI_MIN = CONFIG['guitar_midi_min']
GUITAR_MIDI_MAX = CONFIG['guitar_midi_max']
CHUNK_FRAMES = int(CONFIG['chunk_seconds'] * SAMPLE_RATE) // HOP_LENGTH

# Colab paths
DATA_DIR = Path('/content/guitarset')
SAVE_DIR = Path('/content/guitar_results')
CHECKPOINT_DIR = SAVE_DIR / 'checkpoints'
KONG_PATH = Path('/content/kong_piano_checkpoint.pth')
DRIVE_DIR = Path('/content/drive/MyDrive/stemscribe_checkpoints')


# ============================================================================
# MODEL ARCHITECTURE (Kong CRNN)
# ============================================================================
def init_layer(layer):
    nn.init.xavier_uniform_(layer.weight)
    if hasattr(layer, 'bias') and layer.bias is not None:
        layer.bias.data.fill_(0.)

def init_bn(bn):
    bn.bias.data.fill_(0.)
    bn.weight.data.fill_(1.)

def init_gru(rnn):
    for name, param in rnn.named_parameters():
        if 'weight_ih' in name:
            nn.init.xavier_uniform_(param.data)
        elif 'weight_hh' in name:
            nn.init.orthogonal_(param.data)
        elif 'bias' in name:
            param.data.fill_(0.)


class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.bn2 = nn.BatchNorm2d(out_channels)
        init_bn(self.bn1)
        init_bn(self.bn2)
        init_layer(self.conv1)
        init_layer(self.conv2)

    def forward(self, x, pool_size=(2, 2), pool_type='avg'):
        x = F.relu_(self.bn1(self.conv1(x)))
        x = F.relu_(self.bn2(self.conv2(x)))
        if pool_type == 'avg':
            x = F.avg_pool2d(x, kernel_size=pool_size)
        elif pool_type == 'max':
            x = F.max_pool2d(x, kernel_size=pool_size)
        return x


class AcousticModelCRnn8Dropout(nn.Module):
    def __init__(self, classes_num, midfeat, momentum):
        super().__init__()
        self.conv_block1 = ConvBlock(1, 48)
        self.conv_block2 = ConvBlock(48, 64)
        self.conv_block3 = ConvBlock(64, 96)
        self.conv_block4 = ConvBlock(96, 128)
        self.fc5 = nn.Linear(midfeat, 768, bias=False)
        self.bn5 = nn.BatchNorm1d(768, momentum=momentum)
        init_layer(self.fc5)
        init_bn(self.bn5)
        self.gru = nn.GRU(input_size=768, hidden_size=256, num_layers=2,
                          bias=True, batch_first=True, dropout=0.0, bidirectional=True)
        init_gru(self.gru)
        self.fc = nn.Linear(512, classes_num, bias=True)
        init_layer(self.fc)

    def forward(self, input):
        x = self.conv_block1(input, pool_size=(1, 2), pool_type='avg')
        x = F.dropout(x, p=0.2, training=self.training)
        x = self.conv_block2(x, pool_size=(1, 2), pool_type='avg')
        x = F.dropout(x, p=0.2, training=self.training)
        x = self.conv_block3(x, pool_size=(1, 2), pool_type='avg')
        x = F.dropout(x, p=0.2, training=self.training)
        x = self.conv_block4(x, pool_size=(1, 2), pool_type='avg')
        x = F.dropout(x, p=0.2, training=self.training)
        x = x.transpose(1, 2).flatten(2)
        x = F.relu(self.bn5(self.fc5(x).transpose(1, 2)).transpose(1, 2))
        x = F.dropout(x, p=0.5, training=self.training)
        x, _ = self.gru(x)
        x = self.fc(x)
        return x


class GuitarTranscriptionModel(nn.Module):
    def __init__(self, n_pitches=48):
        super().__init__()
        self.n_pitches = n_pitches
        midfeat = 128 * (N_MELS // 16)
        self.frame_model = AcousticModelCRnn8Dropout(n_pitches, midfeat, momentum=0.01)
        self.onset_model = AcousticModelCRnn8Dropout(n_pitches, midfeat, momentum=0.01)
        self.offset_model = AcousticModelCRnn8Dropout(n_pitches, midfeat, momentum=0.01)
        self.velocity_model = AcousticModelCRnn8Dropout(n_pitches, midfeat, momentum=0.01)
        self.onset_gru = nn.GRU(input_size=n_pitches * 2, hidden_size=n_pitches,
                                num_layers=1, bias=True, batch_first=True, bidirectional=True)
        init_gru(self.onset_gru)
        self.onset_fc = nn.Linear(n_pitches * 2, n_pitches, bias=True)
        init_layer(self.onset_fc)
        self.frame_gru = nn.GRU(input_size=n_pitches * 3, hidden_size=n_pitches,
                                num_layers=1, bias=True, batch_first=True, bidirectional=True)
        init_gru(self.frame_gru)
        self.frame_fc = nn.Linear(n_pitches * 2, n_pitches, bias=True)
        init_layer(self.frame_fc)

    def forward(self, mel):
        x = mel.unsqueeze(1)
        frame_out = self.frame_model(x)
        onset_out = self.onset_model(x)
        offset_out = self.offset_model(x)
        velocity_out = self.velocity_model(x)
        velocity_sigmoid = torch.sigmoid(velocity_out)
        onset_concat = torch.cat([onset_out, onset_out * velocity_sigmoid], dim=2)
        onset_gru_out, _ = self.onset_gru(onset_concat)
        onset_output = self.onset_fc(onset_gru_out)
        onset_sigmoid = torch.sigmoid(onset_output)
        offset_sigmoid = torch.sigmoid(offset_out)
        frame_concat = torch.cat([frame_out, onset_sigmoid, offset_sigmoid], dim=2)
        frame_gru_out, _ = self.frame_gru(frame_concat)
        frame_output = self.frame_fc(frame_gru_out)
        velocity_output = velocity_sigmoid
        return onset_output, frame_output, velocity_output


# ============================================================================
# WEIGHT LOADING
# ============================================================================
def load_kong_weights(model, checkpoint_path):
    state_dict = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    key_mapping = {
        'reg_onset_model': 'onset_model',
        'reg_offset_model': 'offset_model',
        'frame_model': 'frame_model',
        'velocity_model': 'velocity_model',
    }
    loaded = 0
    skipped = 0
    our_state = model.state_dict()
    for kong_key, kong_value in state_dict.items():
        if 'spectrogram' in kong_key or 'logmel' in kong_key or 'bn0' in kong_key:
            skipped += 1
            continue
        if 'reg_onset_gru' in kong_key or 'reg_onset_fc' in kong_key:
            skipped += 1
            continue
        if 'frame_gru' in kong_key or 'frame_fc' in kong_key:
            skipped += 1
            continue
        our_key = kong_key
        for kong_prefix, our_prefix in key_mapping.items():
            if kong_key.startswith(kong_prefix):
                our_key = kong_key.replace(kong_prefix, our_prefix, 1)
                break
        if our_key.endswith('.fc.weight') or our_key.endswith('.fc.bias'):
            parts = our_key.split('.')
            if parts[-2] == 'fc' and 'fc5' not in our_key:
                skipped += 1
                continue
        if our_key in our_state:
            if kong_value.shape == our_state[our_key].shape:
                our_state[our_key] = kong_value
                loaded += 1
            else:
                skipped += 1
        else:
            skipped += 1
    model.load_state_dict(our_state)
    print(f'Kong checkpoint: loaded {loaded} tensors, skipped {skipped}')
    return model


# ============================================================================
# LOSS
# ============================================================================
class FocalBCEWithLogitsLoss(nn.Module):
    def __init__(self, pos_weight=50.0, gamma=2.0):
        super().__init__()
        self.pos_weight = pos_weight
        self.gamma = gamma

    def forward(self, logits, targets):
        pw = torch.tensor([self.pos_weight], device=logits.device, dtype=logits.dtype)
        bce = F.binary_cross_entropy_with_logits(logits, targets, pos_weight=pw, reduction='none')
        probs = torch.sigmoid(logits)
        pt = torch.where(targets > 0.5, probs, 1 - probs)
        focal_weight = (1 - pt) ** self.gamma
        return (focal_weight * bce).mean()


# ============================================================================
# DATASET
# ============================================================================
class GuitarSetDataset(Dataset):
    def __init__(self, track_list, chunk_frames=CHUNK_FRAMES, augment=False):
        self.tracks = track_list
        self.chunk_frames = chunk_frames
        self.augment = augment
        self._cache = {}

    def __len__(self):
        return len(self.tracks) * 8

    def _load_track(self, track_idx):
        if track_idx in self._cache:
            return self._cache[track_idx]
        track = self.tracks[track_idx]
        audio, sr = librosa.load(track['audio'], sr=SAMPLE_RATE, mono=True)
        mel = librosa.feature.melspectrogram(
            y=audio, sr=SAMPLE_RATE, n_fft=CONFIG['n_fft'],
            hop_length=HOP_LENGTH, n_mels=N_MELS
        )
        mel_db = librosa.power_to_db(mel, ref=np.max).T
        jam = jams.load(track['jams'])
        n_frames = mel_db.shape[0]
        onsets = np.zeros((n_frames, N_PITCHES), dtype=np.float32)
        frames = np.zeros((n_frames, N_PITCHES), dtype=np.float32)
        velocities = np.zeros((n_frames, N_PITCHES), dtype=np.float32)
        for ann in jam.annotations:
            if ann.namespace == 'note_midi':
                for obs in ann.data:
                    midi_pitch = int(round(obs.value))
                    if GUITAR_MIDI_MIN <= midi_pitch < GUITAR_MIDI_MAX:
                        pitch_idx = midi_pitch - GUITAR_MIDI_MIN
                        onset_frame = int(obs.time * SAMPLE_RATE / HOP_LENGTH)
                        offset_frame = int((obs.time + obs.duration) * SAMPLE_RATE / HOP_LENGTH)
                        onset_frame = min(onset_frame, n_frames - 1)
                        offset_frame = min(offset_frame, n_frames - 1)
                        onsets[onset_frame, pitch_idx] = 1.0
                        frames[onset_frame:offset_frame + 1, pitch_idx] = 1.0
                        vel = obs.confidence if obs.confidence else 0.8
                        velocities[onset_frame:offset_frame + 1, pitch_idx] = vel
        result = (mel_db, onsets, frames, velocities)
        self._cache[track_idx] = result
        return result

    def __getitem__(self, idx):
        track_idx = idx // 8
        mel_db, onsets, frames, velocities = self._load_track(track_idx)
        n_frames = mel_db.shape[0]
        start = np.random.randint(0, max(1, n_frames - self.chunk_frames))
        end = start + self.chunk_frames
        mel_chunk = mel_db[start:end]
        onset_chunk = onsets[start:end]
        frame_chunk = frames[start:end]
        vel_chunk = velocities[start:end]
        if mel_chunk.shape[0] < self.chunk_frames:
            pad = self.chunk_frames - mel_chunk.shape[0]
            mel_chunk = np.pad(mel_chunk, ((0, pad), (0, 0)))
            onset_chunk = np.pad(onset_chunk, ((0, pad), (0, 0)))
            frame_chunk = np.pad(frame_chunk, ((0, pad), (0, 0)))
            vel_chunk = np.pad(vel_chunk, ((0, pad), (0, 0)))
        if self.augment:
            mel_chunk = self._augment_mel(mel_chunk)
        return (
            torch.from_numpy(mel_chunk.copy()),
            torch.from_numpy(onset_chunk.copy()),
            torch.from_numpy(frame_chunk.copy()),
            torch.from_numpy(vel_chunk.copy()),
        )

    def _augment_mel(self, mel):
        mel = mel.copy()
        if random.random() < 0.7:
            mel = mel + random.uniform(*CONFIG['gain_range'])
        if random.random() < 0.5:
            snr_db = random.uniform(*CONFIG['noise_snr_range'])
            noise_level = 10 ** (-snr_db / 20) * 10
            mel = mel + np.random.randn(*mel.shape).astype(np.float32) * noise_level
        for _ in range(CONFIG['spec_augment_freq_masks']):
            if random.random() < 0.5:
                fw = random.randint(1, CONFIG['spec_augment_freq_width'])
                fs = random.randint(0, max(0, mel.shape[1] - fw))
                mel[:, fs:fs + fw] = mel.min()
        for _ in range(CONFIG['spec_augment_time_masks']):
            if random.random() < 0.5:
                tw = random.randint(1, CONFIG['spec_augment_time_width'])
                ts = random.randint(0, max(0, mel.shape[0] - tw))
                mel[ts:ts + tw, :] = mel.min()
        return mel


# ============================================================================
# EVALUATION
# ============================================================================
def evaluate_onset_f1(model, val_loader, device, threshold=0.5):
    model.eval()
    all_f1, all_prec, all_rec = [], [], []
    with torch.no_grad():
        for mel, onset_targets, _, _ in val_loader:
            mel = mel.to(device)
            onset_targets = onset_targets.to(device)
            onset_logits, _, _ = model(mel)
            onset_probs = torch.sigmoid(onset_logits)
            for i in range(onset_probs.shape[0]):
                pred = (onset_probs[i] > threshold).cpu().numpy()
                ref = onset_targets[i].cpu().numpy()
                pred_times = np.where(pred.any(axis=1))[0].astype(float) * HOP_LENGTH / SAMPLE_RATE
                ref_times = np.where(ref.any(axis=1))[0].astype(float) * HOP_LENGTH / SAMPLE_RATE
                if len(ref_times) == 0:
                    continue
                if len(pred_times) == 0:
                    all_f1.append(0.0); all_prec.append(0.0); all_rec.append(0.0)
                    continue
                p, r, f = mir_eval.onset.f_measure(ref_times, pred_times, window=0.05)
                all_f1.append(f); all_prec.append(p); all_rec.append(r)
    if not all_f1:
        return 0.0, 0.0, 0.0
    return np.mean(all_f1), np.mean(all_prec), np.mean(all_rec)


def compute_batch_f1(onset_logits, onset_targets, threshold=0.5):
    with torch.no_grad():
        preds = (torch.sigmoid(onset_logits) > threshold).float()
        tp = (preds * onset_targets).sum()
        fp = (preds * (1 - onset_targets)).sum()
        fn = ((1 - preds) * onset_targets).sum()
        precision = tp / (tp + fp + 1e-8)
        recall = tp / (tp + fn + 1e-8)
        f1 = 2 * precision * recall / (precision + recall + 1e-8)
        return f1.item(), precision.item(), recall.item()


# ============================================================================
# TRACK LOADING
# ============================================================================
def load_guitarset_tracks():
    audio_dir = DATA_DIR / 'audio_mono-mic'
    annot_dir = DATA_DIR / 'annotation'
    tracks = []
    for audio_path in sorted(audio_dir.glob('*.wav')):
        stem = audio_path.stem
        jams_stem = stem.replace('_mic', '').replace('_mix', '')
        jams_path = annot_dir / f'{jams_stem}.jams'
        if not jams_path.exists():
            for suffix in ['_mic', '_mix', '_hex_cln', '_hex']:
                test_stem = stem.split(suffix)[0] if suffix in stem else stem
                test_path = annot_dir / f'{test_stem}.jams'
                if test_path.exists():
                    jams_path = test_path
                    break
        if jams_path.exists():
            tracks.append({
                'audio': str(audio_path),
                'jams': str(jams_path),
                'player': audio_path.stem[:2],
                'name': audio_path.stem,
            })
    print(f'Loaded {len(tracks)} tracks')
    return tracks


# ============================================================================
# TRAINING
# ============================================================================
def train_phase(model, train_loader, val_loader, device, phase, epochs, lr,
                freeze_cnn=False, best_val_f1=0.0, start_epoch=0):
    print(f'\n{"="*60}')
    print(f'PHASE {phase}: {"Frozen CNN" if freeze_cnn else "Full Fine-tuning"}')
    print(f'Epochs: {epochs}, LR: {lr:.2e}')
    print(f'{"="*60}\n')

    if freeze_cnn:
        for name, param in model.named_parameters():
            if any(x in name for x in ['conv_block', 'bn1', 'bn2', 'conv1', 'conv2']):
                if any(m in name for m in ['frame_model', 'onset_model', 'offset_model', 'velocity_model']):
                    param.requires_grad = False

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f'Trainable: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)')

    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    onset_loss_fn = FocalBCEWithLogitsLoss(pos_weight=CONFIG['onset_pos_weight'], gamma=CONFIG['focal_gamma'])
    frame_loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([CONFIG['frame_pos_weight']]).to(device))
    velocity_loss_fn = nn.MSELoss()

    collapse_counter = 0

    for epoch in range(start_epoch, epochs):
        t0 = time.time()
        model.train()
        running_loss = 0.0
        num_batches = 0
        epoch_onset_logits = []
        epoch_onset_targets = []

        pbar = tqdm(train_loader, desc=f'P{phase} Epoch {epoch+1}/{epochs}')
        for mel, onset_tgt, frame_tgt, vel_tgt in pbar:
            mel = mel.to(device)
            onset_tgt = onset_tgt.to(device)
            frame_tgt = frame_tgt.to(device)
            vel_tgt = vel_tgt.to(device)
            onset_logits, frame_logits, vel_pred = model(mel)
            loss_onset = onset_loss_fn(onset_logits, onset_tgt)
            loss_frame = frame_loss_fn(frame_logits, frame_tgt)
            onset_mask = onset_tgt > 0.5
            loss_vel = velocity_loss_fn(vel_pred[onset_mask], vel_tgt[onset_mask]) if onset_mask.any() else torch.tensor(0.0, device=device)
            loss = loss_onset + loss_frame + 0.5 * loss_vel
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            running_loss += loss.item()
            num_batches += 1
            pbar.set_postfix(loss=f'{loss.item():.4f}')
            if num_batches % 5 == 0:
                epoch_onset_logits.append(onset_logits.detach().cpu())
                epoch_onset_targets.append(onset_tgt.detach().cpu())

        scheduler.step()
        train_loss = running_loss / max(num_batches, 1)
        if epoch_onset_logits:
            train_f1, train_prec, train_rec = compute_batch_f1(
                torch.cat(epoch_onset_logits), torch.cat(epoch_onset_targets))
        else:
            train_f1, train_prec, train_rec = 0, 0, 0

        # Validate (mir_eval every 5 epochs)
        if (epoch + 1) % 5 == 0 or epoch == epochs - 1:
            val_f1, val_prec, val_rec = evaluate_onset_f1(model, val_loader, device)
            eval_type = 'mir_eval'
        else:
            model.eval()
            vl, vt = [], []
            with torch.no_grad():
                for mel, onset_tgt, _, _ in val_loader:
                    ol, _, _ = model(mel.to(device))
                    vl.append(ol.cpu()); vt.append(onset_tgt)
            val_f1, val_prec, val_rec = compute_batch_f1(torch.cat(vl), torch.cat(vt))
            eval_type = 'batch'

        elapsed = time.time() - t0
        print(f'\nP{phase} Epoch {epoch+1}/{epochs} ({elapsed:.0f}s) LR={scheduler.get_last_lr()[0]:.2e}')
        print(f'  Train - Loss: {train_loss:.4f} | F1: {train_f1:.4f} | P: {train_prec:.4f} | R: {train_rec:.4f}')
        print(f'  Val   - F1: {val_f1:.4f} | P: {val_prec:.4f} | R: {val_rec:.4f} [{eval_type}]')

        # Collapse detection
        if val_rec < 0.01 and epoch > 3:
            collapse_counter += 1
            print(f'  WARNING: Recall near zero ({collapse_counter}/10)')
            if collapse_counter >= 10:
                print('COLLAPSED. Stopping.')
                break
        else:
            collapse_counter = 0

        # Save best
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            save_path = str(SAVE_DIR / 'best_guitar_model.pt')
            torch.save({
                'epoch': epoch, 'phase': phase,
                'model_state_dict': model.state_dict(),
                'val_f1': val_f1, 'val_prec': val_prec, 'val_rec': val_rec,
                'config': CONFIG,
            }, save_path)
            print(f'  -> New best! F1={val_f1:.4f}')
            # Sync to Drive
            if DRIVE_DIR.exists():
                shutil.copy2(save_path, str(DRIVE_DIR / 'best_guitar_model.pt'))
                print(f'  -> Synced to Drive')

        # Periodic Drive checkpoint
        if (epoch + 1) % 5 == 0 and DRIVE_DIR.exists():
            ckpt_path = str(CHECKPOINT_DIR / f'phase{phase}_epoch{epoch+1}.pt')
            torch.save({
                'epoch': epoch, 'phase': phase,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_val_f1': best_val_f1, 'config': CONFIG,
            }, ckpt_path)
            shutil.copy2(ckpt_path, str(DRIVE_DIR / f'phase{phase}_epoch{epoch+1}.pt'))

    for param in model.parameters():
        param.requires_grad = True
    return best_val_f1


# ============================================================================
# MAIN
# ============================================================================
def main():
    print('=' * 60)
    print('Guitar Transcription Training - Colab')
    print('Kong Piano -> GuitarSet Fine-tuning')
    print('=' * 60)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    if torch.cuda.is_available():
        print(f'GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB)')

    tracks = load_guitarset_tracks()
    val_player = '05'
    train_tracks = [t for t in tracks if t['player'] != val_player]
    val_tracks = [t for t in tracks if t['player'] == val_player]
    print(f'Train: {len(train_tracks)}, Val: {len(val_tracks)} (player {val_player})')

    train_dataset = GuitarSetDataset(train_tracks, augment=CONFIG['augment'])
    val_dataset = GuitarSetDataset(val_tracks, augment=False)
    train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'],
                              shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
    val_loader = DataLoader(val_dataset, batch_size=CONFIG['batch_size'],
                            shuffle=False, num_workers=2, pin_memory=True)

    model = GuitarTranscriptionModel(n_pitches=N_PITCHES).to(device)
    print(f'Model params: {sum(p.numel() for p in model.parameters()):,}')

    model = load_kong_weights(model, str(KONG_PATH))

    best_f1 = 0.0
    best_f1 = train_phase(model, train_loader, val_loader, device,
                          phase=1, epochs=CONFIG['phase1_epochs'], lr=CONFIG['phase1_lr'],
                          freeze_cnn=True, best_val_f1=best_f1)

    best_f1 = train_phase(model, train_loader, val_loader, device,
                          phase=2, epochs=CONFIG['phase2_epochs'], lr=CONFIG['phase2_lr'],
                          freeze_cnn=False, best_val_f1=best_f1)

    # Final eval
    best_ckpt = torch.load(str(SAVE_DIR / 'best_guitar_model.pt'), map_location=device, weights_only=False)
    model.load_state_dict(best_ckpt['model_state_dict'])
    f1, prec, rec = evaluate_onset_f1(model, val_loader, device)
    print(f'\n{"="*60}')
    print(f'FINAL: Onset F1={f1:.4f} P={prec:.4f} R={rec:.4f}')
    print(f'Target: 0.87+ | Improvement over Basic Pitch (0.79): +{(f1-0.79)*100:.1f}pp')
    print(f'Model: {SAVE_DIR / "best_guitar_model.pt"}')
    if DRIVE_DIR.exists():
        print(f'Drive: {DRIVE_DIR / "best_guitar_model.pt"}')
    print(f'{"="*60}')

    with open(SAVE_DIR / 'training_summary.json', 'w') as f:
        json.dump({'f1': f1, 'precision': prec, 'recall': rec, 'config': CONFIG}, f, indent=2, default=str)


if __name__ == '__main__':
    main()

In [ ]:
# Cell 8: RUN TRAINING
# This takes ~8 hours on A100 (2h Phase 1 + 6h Phase 2)
# Checkpoints are saved to Google Drive every 5 epochs
!python /content/train_guitar_colab.py

In [ ]:
# Cell 9: After training — check results
import json
with open('/content/guitar_results/training_summary.json') as f:
    summary = json.load(f)
print(f'Final Onset F1: {summary["f1"]:.4f}')
print(f'Precision: {summary["precision"]:.4f}')
print(f'Recall: {summary["recall"]:.4f}')
print(f'\nModel saved to Google Drive: /content/drive/MyDrive/stemscribe_checkpoints/best_guitar_model.pt')
print('\nTo use in StemScribe:')
print('  1. Download best_guitar_model.pt from Google Drive')
print('  2. Copy to stemscribe/backend/models/pretrained/best_guitar_model.pt')
print('  3. Create guitar_nn_transcriber.py following drum_nn_transcriber.py pattern')

In [ ]:
# Cell 10: Download model to local machine (alternative to Drive)
from google.colab import files
files.download('/content/guitar_results/best_guitar_model.pt')